# C12-classical-models — Practice p27 — Solution


Broadcasting constructs every support/probe squared distance at once; signed coefficients are applied only after the unsigned RBF matrix is available for audit.


In [ ]:
import numpy as np

support_vectors_p27 = np.array([[-1.,0.],[0.,1.],[1.,0.],[0.,-1.]], dtype=np.float64)
signed_dual_p27 = np.array([-0.8, 0.6, 0.7, -0.5], dtype=np.float64)
intercept_p27 = 0.15
gamma_p27 = 0.5
X_probe_p27 = np.array([[0.,0.],[1.,1.],[-1.,-1.]], dtype=np.float64)


def reconstruct_scores(support_vectors, signed_dual, intercept, gamma, X_probe):
    support_vectors=np.asarray(support_vectors,dtype=np.float64)
    signed_dual=np.asarray(signed_dual,dtype=np.float64)
    X_probe=np.asarray(X_probe,dtype=np.float64)
    if support_vectors.ndim!=2 or X_probe.ndim!=2 or signed_dual.ndim!=1:
        raise ValueError("invalid dimensions")
    if signed_dual.shape!=(support_vectors.shape[0],) or X_probe.shape[1]!=support_vectors.shape[1]:
        raise ValueError("shape mismatch")
    if not all(np.isfinite(a).all() for a in (support_vectors,signed_dual,X_probe)) or not np.isfinite(intercept) or not np.isfinite(gamma) or gamma<=0:
        raise ValueError("invalid values")
    squared=((support_vectors[:,None,:]-X_probe[None,:,:])**2).sum(axis=2)
    kernel_matrix=np.exp(-float(gamma)*squared).astype(np.float64)
    contributions=(signed_dual[:,None]*kernel_matrix).astype(np.float64)
    scores=(contributions.sum(axis=0)+float(intercept)).astype(np.float64)
    predictions=np.where(scores>=0.0,1,-1).astype(np.int64)
    return {"kernel_matrix":kernel_matrix,"contributions":contributions,
            "scores":scores,"predictions":predictions}


result_p27 = reconstruct_scores(
    support_vectors_p27, signed_dual_p27, intercept_p27, gamma_p27, X_probe_p27
)
first_probe_derivation_p27 = "Every first-probe squared distance is 1, so every kernel is exp(-1/2). The signed coefficients sum to -0.8+0.6+0.7-0.5=0, giving score 0.15+0*exp(-1/2)=0.15."
support_audit_p27 = '''Coefficient signs infer support labels (-,+,+,-). Every supplied coefficient is nonzero, so every RBF contribution is nonzero at finite probes. A non-support row has coefficient zero and vanishes from the decision sum.'''


### Answer check


In [ ]:
ATOL=1e-12
RTOL=1e-10
expected_scores_p27=np.array([0.14999999999999994,0.831779359415355,-0.531779359415355])
assert set(result_p27)=={"kernel_matrix","contributions","scores","predictions"}
assert result_p27["kernel_matrix"].shape==(4,3) and result_p27["kernel_matrix"].dtype==np.float64
assert np.allclose(result_p27["kernel_matrix"][:,0],np.full(4,np.exp(-0.5)),atol=ATOL,rtol=RTOL)
assert np.allclose(result_p27["contributions"][:,0],np.exp(-0.5)*signed_dual_p27,atol=ATOL,rtol=RTOL)
assert np.allclose(result_p27["scores"],expected_scores_p27,atol=ATOL,rtol=RTOL)
assert np.array_equal(result_p27["predictions"],[1,1,-1])
assert "exp(-1/2)" in first_probe_derivation_p27 and "vanishes" in support_audit_p27
